# Lunar-HFE: run everything

This notebook reproduces the whole study behind the JGR: Planets letter
*"Difference of the Lunar Regolith Thermal Conductivity $K_d$ at the Apollo 15
and 17 Heat-Flow Boreholes from Global Model Values"* and its Supporting
Information (SI), code release **v2.0-jgr**.

**How to use it:** choose `MODE` in the first code cell below, then
**Run All** (JupyterLab: *Run ▸ Run All Cells*; VS Code: *Run All*).

| `MODE` | What runs | Time |
|---|---|---|
| `"show"` (default) | nothing is recomputed: every step reads its stored result in `code/results/` | a few minutes |
| `"quick"` | the short steps run; the long ones run their fast form (re-analysing stored model runs) or show their stored result | about 45 min |
| `"full"` | every step from scratch, as for the paper | about 6 h on a six-core laptop |

**What it contains**

1. **Setup**: checks that the computer is ready; optional downloads.
2. **The analysis**: every step of the study, in the order in which the steps need each other.
3. **The figures**: every figure of the letter and the SI.
4. **The paper, section by section**: each number of the letter next to the value
   this repository gives (✓ / ✗), in the order of the letter: the models against each
   sensor, one measurement at a time, all measurements together, the checks, and the
   discussion.
5. **Summary**: how many values were checked, and whether all of them match.

Each step is one script in `code/pipeline/`. The cell runs it and shows its
printout as it goes. The notebook computes nothing itself, so it and
`make aux` run exactly the same code. On a Mac every step keeps the computer
awake (`caffeinate`), but not with the lid closed: keep it open during a long run.

Two optional notebooks go deeper: `optional_anchor_method.ipynb` (how the
solver reaches its steady state, step by step) and `optional_performance.ipynb`
(the solver in Python against C++).

In [ ]:
# How much to run:  "show"  (minutes) | "quick"  (about 45 min) | "full"  (about 6 h)
MODE = "show"

try:
    from lunar import notebook as nb
except ImportError:
    raise ImportError("The lunar package is not installed. In a terminal, in the repository folder, run:\n"
                      "    pip install -e \".[dev,mcmc]\"\nthen restart the kernel.")
import numpy as np
import pandas as pd
from IPython.display import display
nb.set_mode(MODE)
S2 = ("A15", "A17")
DIVINER = ("data/diviner/gcp/global_cumul_avg_cyl_20n30n_002.tab", "data/diviner/gcp/global_cumul_avg_cyl_10n20n_002.tab")

---
# 1. Setup

## 1.1 Python and the core libraries
Python 3.10 or newer. A version problem is the most common reason a notebook
will not run, so it is checked first.

In [ ]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()} {platform.machine()}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"
import scipy, matplotlib, numba
for mod in (np, scipy, matplotlib, pd, numba):
    print(f"  {mod.__name__:<12} {mod.__version__}")

## 1.2 The `lunar` package and its constants
The physics is the installed package `lunar` (`code/src/lunar/`). This checks
the constants the study is defined by: the Stefan–Boltzmann constant and the
Hayne et al. (2017) regolith parameters.

In [ ]:
import math
from lunar import constants as c
assert math.isclose(c.SIGMA_SB, 5.670374419e-8, rel_tol=1e-6), "SIGMA_SB constant wrong"
assert math.isclose(c.K_DEEP, 3.4e-3, rel_tol=1e-6), "Hayne K_d default is wrong"
assert math.isclose(c.K_SURFACE, 7.4e-4, rel_tol=1e-6), "Hayne K_s default is wrong"
assert math.isclose(c.H_PARAMETER, 0.06, rel_tol=1e-6), "Hayne H is wrong"
print(f"  sigma = {c.SIGMA_SB} W m^-2 K^-4")
print(f"  Hayne global K_d = {c.K_DEEP*1e3:.1f} mW m^-1 K^-1,  K_s = {c.K_SURFACE*1e3:.2f},  H = {c.H_PARAMETER*100:.0f} cm")

## 1.3 The Apollo heat-flow record
The 1971–1977 record (St. Clair et al., 2020; Nagihara et al., 2018) is bundled
in `code/data/apollo/`. Only the sensors below 80 cm are fitted: 7 at Apollo 15
and 16 at Apollo 17.

In [ ]:
from lunar.apollo_helpers import extract_sensor_stability
for mission, expected in (("a15", 7), ("a17", 16)):
    n_deep = int((extract_sensor_stability(mission, min_depth_cm=80)["depth_cm_all"] >= 80).sum())
    print(f"  Apollo {mission[-2:]}: {n_deep} sensors below 80 cm (expected {expected})")
    assert n_deep == expected, f"sensor count mismatch for {mission}"

## 1.4 Solar geometry (SPICE)
The real-forcing run of the annual wave needs the Sun's position from the
bundled JPL DE440 kernels.

In [ ]:
from lunar.ephem import et_from_iso, solar_vector_moon_frame
sv = solar_vector_moon_frame(et_from_iso(["2024-01-01T00:00:00"]))
assert abs(np.linalg.norm(sv[0]) - 1.0) < 1e-6, "should be a unit vector"
print("  SPICE kernels OK")

## 1.5 Optional downloads
Two public data sets are too large for the repository. They are needed only to
**recompute** the steps that use them; everything else works without them.

| Data | Size | Needed for |
|---|---|---|
| Diviner surface temperatures (PDS) | ~310 MB | rerunning the Diviner comparison (SI Table S13, SI Figure S6) |
| LOLA elevation map, 16 pixels per degree (PDS) | 33 MB | SI Figure S1 (the landing-site map) |

Set `DOWNLOAD = True` and run the cell to fetch whatever is missing.

In [ ]:
DOWNLOAD = False

from lunar._bootstrap import _download
DIV = [nb.CODE / f for f in DIVINER]
LOLA = nb.CODE / "data/lola/ldem_16.img"
LOLA_URL = ("https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/"
            "data/lola_gdr/cylindrical/img/ldem_16.img")
print(f"  Diviner tiles : {'present' if all(f.exists() for f in DIV) else 'not downloaded'}")
print(f"  LOLA map      : {'present' if LOLA.exists() else 'not downloaded'}")
if DOWNLOAD:
    if not all(f.exists() for f in DIV):
        nb.run("fetch_diviner.py")
    if not LOLA.exists():
        LOLA.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", LOLA_URL)
        assert _download(LOLA_URL, LOLA, min_bytes=30_000_000), "LOLA download failed"
        print("  LOLA map downloaded")

---
# 2. The analysis

Every step of the study, in the order in which the steps need each other (the
same order as `make retrieve aux`). What runs depends on `MODE`; a step that
does not run says which stored result it uses. Part 4 then reads the results
in the order of the letter.

## 2.1 The data and the earlier, fixed-albedo analyses
`retrieve_kd.py` is the study's first stage (the albedo held fixed). The later
steps reuse its sensor handling and outputs, and the fixed-albedo results
appear in Section 3.2 of the letter. `compute_stability_windows.py` lists every
sensor's stability window (SI Table S1).

In [ ]:
nb.step("compute/retrieve_kd.py", minutes=95, writes="kd_retrieval_results.json")
nb.step("compute/compute_common_epoch.py", minutes=15, writes="common_epoch_sensitivity.json")
nb.step("compute/compute_diviner_closure.py", minutes=3, writes="diviner_closure.json", needs=DIVINER)
nb.step("compute/compute_review_diagnostics.py", minutes=5, writes="review_diagnostics.json", needs=DIVINER)
nb.step("compute/compute_albedo_diagnostics.py", minutes=13, writes="albedo_diagnostics.json")
nb.step("compute/compute_albedo_sensitivity.py", minutes=35, writes="albedo_sensitivity.json")
nb.step("compute/compute_albedo_anchor.py", minutes=5, writes="albedo_anchor.json")
nb.step("compute/compute_stability_windows.py", minutes=0.2, writes="stability_windows.json")

## 2.2 The joint fit of the albedo and $K_d$
The main result (letter Sections 2.3, 3.3). At each site the model is solved
for 25 albedos × 94 values of $K_d$ and both are fitted to the sensor
temperatures, the mean surface temperature and the annual-wave diffusivity.
A full run takes about 25 min; "quick" re-analyses the stored grid of model
runs (`joint_albedo_fit_cache.npz`) in seconds, with the same minimum and the
same 1500-draw bootstrap. Then: the solver tests at the fit (SI Table S3), the
worked example of the trade-off (SI Table S11), each probe's own diffusivity
and the ±5 K Apollo test (SI Table S9), and the bootstrap with the sensors
grouped by probe.

In [ ]:
nb.step("compute/compute_joint_albedo_fit.py", minutes=25, writes="joint_albedo_fit.json",
        quick=("--reuse",), quick_needs="results/joint_albedo_fit_cache.npz")
nb.step("compute/certify_joint_numerics.py", minutes=3, writes="joint_numerics_certification.json")
nb.step("compute/compute_joint_valley.py", minutes=0.2, writes="joint_valley.json")
nb.step("compute/compute_joint_probe_checks.py", minutes=0.3, writes="joint_probe_checks.json")
nb.step("compute/compute_joint_block_bootstrap.py", minutes=1, writes="joint_block_bootstrap.json")

## 2.3 Uncertainty and tests
The error budget (Table 2; about 80 min, or the fast form when
`results/joint_fit_sensitivities_cache.npz` is on your computer: it is not on
GitHub, 27 MB), the model at the fit against each sensor with the probe and
thermometer-type splits (Table 1, Figure 3), the Diviner comparison, the
measured gradients, the radiative coefficient and the density (SI Table S14;
about 50 min) and the likelihood-ratio tests.

In [ ]:
nb.step("compute/compute_joint_fit_sensitivities.py", minutes=80, writes="joint_fit_sensitivities.json",
        quick=("--reuse",), quick_needs="results/joint_fit_sensitivities_cache.npz")
nb.step("compute/compute_joint_fit_checks.py", minutes=4, writes="joint_fit_checks.json")
nb.step("compute/compute_joint_diviner.py", minutes=3, writes="joint_diviner.json", needs=DIVINER)
nb.step("compute/compute_gradient_epochs.py", minutes=0.2, writes="gradient_epochs.json")
nb.step("compute/compute_joint_chi_density.py", minutes=50, writes="joint_chi_density.json")
nb.step("compute/compute_likelihood_ratio.py", minutes=0.2, writes="likelihood_ratio.json")

## 2.4 The annual wave, the warming after deployment, and the posterior
The annual wave measured in the full record and modeled with the real solar
forcing (Figure 5, SI Table S10), the joint fit with our own diffusivities, a
different near-surface layer against the annual wave, the darkening of the
surface at deployment (SI Table S8), and the posterior with the basal heat flow
and the density also free (Figure 6, SI Table S6). The posterior solves a grid
of models first (about 2 h, resumable); "quick" samples the stored grid
(`joint_mcmc_grid.npz`) in about a minute.

In [ ]:
nb.step("compute/compute_annual_wave.py", minutes=3, writes="annual_wave.json")
nb.step("compute/compute_own_diffusivity_fit.py", minutes=0.3, writes="own_diffusivity_fit.json")
nb.step("compute/compute_annual_wave_surface_test.py", minutes=6, writes="annual_wave_surface_test.json")
nb.step("compute/compute_transient_warming.py", minutes=2, writes="transient_warming.json")
nb.step("compute/compute_joint_mcmc.py", minutes=120, writes="joint_mcmc.json",
        quick=("--sample",), quick_needs="results/joint_mcmc_grid.npz")

---
# 3. The figures

`make_joint_figures.py` draws Figures 1–7 of the letter and SI Figures S3–S5
(about 4 min). SI Figure S1 needs the LOLA map, SI Figure S2 needs `pdflatex`,
and SI Figure S6 is drawn by the Diviner step (it needs the Diviner tiles). In
"show" mode a figure is drawn only if it is missing; a fresh download has none.

In [ ]:
nb.figures("figures/make_joint_figures.py", minutes=4,
           makes=("fig_joint_intro_column", "fig_joint_apollo_timeline_probes", "fig_joint_sensors",
                  "fig_joint_constraints", "fig_joint_annual_wave", "fig_joint_mcmc", "fig_joint_variants",
                  "fig_joint_amplitude_vs_depth", "fig_joint_anchor_process", "fig_joint_robustness"))
nb.figures("figures/make_context_map_figure.py", makes="fig_context_map", minutes=1, needs=("data/lola/ldem_16.img",))
nb.figures("figures/make_study_overview.py", makes="fig_study_overview", minutes=0.5)
nb.figures("compute/compute_joint_diviner.py", makes="fig_joint_diviner_closure", minutes=3, needs=DIVINER)
PAPER_FIGURES = {
    "Figure 1": "fig_joint_intro_column", "Figure 2": "fig_joint_apollo_timeline_probes",
    "Figure 3": "fig_joint_sensors", "Figure 4": "fig_joint_constraints", "Figure 5": "fig_joint_annual_wave",
    "Figure 6": "fig_joint_mcmc", "Figure 7": "fig_joint_variants",
    "SI Figure S1": "fig_context_map", "SI Figure S2": "fig_study_overview",
    "SI Figure S3": "fig_joint_amplitude_vs_depth", "SI Figure S4": "fig_joint_anchor_process",
    "SI Figure S5": "fig_joint_robustness", "SI Figure S6": "fig_joint_diviner_closure"}
for k, f in PAPER_FIGURES.items():
    print(f"  {'✓' if (nb.FIGURES / f'{f}.pdf').exists() else '✗ not made'}  {k:13s} figures/{f}.pdf")

---
# 4. The paper, section by section

Each table puts a value from `code/results/` next to the number printed in the
letter, rounded the way the letter rounds it. ✓ means they agree.

In [ ]:
R = {n: nb.load(f"{n}.json") for n in (
    "stability_windows", "joint_albedo_fit", "joint_fit_checks", "joint_fit_sensitivities",
    "joint_numerics_certification", "likelihood_ratio", "gradient_epochs", "albedo_sensitivity",
    "albedo_anchor", "joint_probe_checks", "joint_valley", "joint_block_bootstrap", "annual_wave",
    "annual_wave_surface_test", "own_diffusivity_fit", "transient_warming", "joint_mcmc", "joint_chi_density")}
JF, FC, LR, GE = R["joint_albedo_fit"], R["joint_fit_checks"], R["likelihood_ratio"], R["gradient_epochs"]
best = {s: JF["sites"][s]["with_diffusivity"]["best"] for s in S2}
print("results loaded:", ", ".join(R))

## Section 2.1: the sensors and their stability windows (Figure 2, SI Table S1, SI Figure S3)
Each sensor's equilibrium temperature is its mean over a *stability window* at
the end of its record. The window is the earliest start whose trailing trend is
below 0.08 K/yr (the strict test); if none passes, the last 25% of the samples.
Sensors in the top 80 cm, disturbed by the borestem, are listed but not fitted.

In [ ]:
win = R["stability_windows"]
for s in S2:
    df = pd.DataFrame(win[s])
    print(f"Apollo {s[1:]}: {int(df['used'].sum())} sensors used of {len(df)}")
    display(df[["probe", "sensor", "depth_cm", "used", "method", "window_start", "window_end", "T_eq_K",
                "trailing_slope_K_per_yr"]].round(3))

In [ ]:
chk = nb.Checks("Section 2.1, SI Table S1:")
fallback = []
for s, n, strict in (("A15", "7", "3"), ("A17", "16", "6")):
    used = [r for r in win[s] if r["used"]]
    chk.add_text(f"{s}: sensors below 80 cm", str(len(used)), n)
    chk.add_text(f"{s}: windows that pass the strict test", str(sum(not r["method"].startswith("fallback") for r in used)), strict)
    fallback += [r["trailing_slope_K_per_yr"] for r in used if r["method"].startswith("fallback")]
chk.add_text("fallback windows: still warming by (K/yr)", nb.span(fallback, ".2f"), "0.11-0.47")
chk.add_text("sensors still warming at the end of the record (Sec. 4.4)", str(len(fallback)), "14")
chk.show()

**Figure 2.** The sensor records and their stability windows.

In [ ]:
nb.show("fig_joint_apollo_timeline_probes", width=700)

**SI Figure S3.** The daily temperature swing against depth: why the
sensors above 80 cm are left out.

In [ ]:
nb.show("fig_joint_amplitude_vs_depth")

## Section 2.2: the thermal model (Figure 1, SI Tables S3, S4, SI Figures S1, S2, S4)
Every fixed input lives in `code/src/lunar/config.py` (SI Table S4). The albedo
and $K_d$ are not inputs: they are fitted. The solver tests (SI Table S3)
repeat the last step of the joint fit with one numerical setting changed at a
time.

In [ ]:
from lunar.config import SITES, HAYNE, GRID, DT_STEP
from lunar.constants import RHO_SURFACE, RHO_DEEP
rows = [("latitude (deg N)", SITES["A15"]["lat"], SITES["A17"]["lat"]),
        ("emissivity", SITES["A15"]["emissivity"], SITES["A17"]["emissivity"]),
        ("basal heat flow Q_b (mW m^-2)", SITES["A15"]["Q_BASAL"] * 1e3, SITES["A17"]["Q_BASAL"] * 1e3),
        ("surface conductivity K_s (mW m^-1 K^-1)", HAYNE["K_S"] * 1e3, HAYNE["K_S"] * 1e3),
        ("compaction depth H (cm)", HAYNE["H"] * 100, HAYNE["H"] * 100),
        ("radiative coefficient chi", HAYNE["CHI"], HAYNE["CHI"]),
        ("surface / deep density (kg m^-3)", f"{RHO_SURFACE:.0f} / {RHO_DEEP:.0f}", f"{RHO_SURFACE:.0f} / {RHO_DEEP:.0f}")]
display(pd.DataFrame(rows, columns=["input (SI Table S4)", "Apollo 15", "Apollo 17"]))
print(f"grid: {GRID}   time step: {DT_STEP:.0f} s")
cert = R["joint_numerics_certification"]["sites"]
display(pd.DataFrame({s: {k: f"{v:+.5f}" for k, v in cert[s]["delta_kd_mW"].items()} for s in S2})
        .rename_axis("SI Table S3: change of K_d* (mW m^-1 K^-1)"))

In [ ]:
chk = nb.Checks("Section 2.2, SI Tables S3, S4:")
chk.add("A15: Q_b (mW m^-2)", SITES["A15"]["Q_BASAL"] * 1e3, "21", ".0f")
chk.add("A17: Q_b (mW m^-2)", SITES["A17"]["Q_BASAL"] * 1e3, "16", ".0f")
chk.add("deep density (kg m^-3)", RHO_DEEP, "1800", ".0f")
chk.add("chi", HAYNE["CHI"], "2.7", ".1f")
mx = lambda key: max(cert[s]["summary"][key] for s in S2)
chk.add("SI Table S3: length of each shallow run", mx("n_inner_max"), "0.0002", ".4f")
chk.add("SI Table S3: Step A on the full column", mx("full_column"), "0.0006", ".4f")
chk.add("SI Table S3: time step halved", mx("dt_per_halving_max"), "0.001", ".3f")
chk.add("SI Table S3: finer grid", mx("grid_fine"), "0.0024", ".4f")
chk.add("SI Table S3: anchor depth", mx("anchor_max"), "0.0001", ".4f")
chk.add("SI Table S3: K_d spacing", mx("kd_spacing"), "0.0003", ".4f")
chk.add("grid, time step or anchor depth: K_d moves by less than",
        max(mx("grid_fine"), mx("dt_per_halving_max"), mx("anchor_max")), "<0.003")
chk.add("starting temperature: profiles change by about (K)", mx("start_profile_max_K"), "1e-5", ".0e")
chk.show()

**Figure 1.** The model column, the probes and the two conductivity models.

In [ ]:
nb.show("fig_joint_intro_column")

**SI Figure S1** (needs the LOLA map), **SI Figure S2** (the study at a
glance) and **SI Figure S4** (the solver at the joint fit, step by step).

In [ ]:
for f in ("fig_context_map", "fig_study_overview", "fig_joint_anchor_process"):
    nb.show(f, width=750)

## Section 2.3: fitting $K_d$
The misfit $J$ (Equation 11) is evaluated on 25 albedos × 94 values of $K_d$;
the bootstrap repeats the fit 1500 times with the sensors resampled, each depth
jittered by 2.5 cm and the surface mean and diffusivities redrawn. Its 95%
intervals agree with those of the profile likelihood, and grouping the sensors
by probe hardly changes them (SI Table S5).

In [ ]:
m = JF["meta"]
bb = R["joint_block_bootstrap"]["schemes"]
prof = max(abs(JF["sites"][s]["with_diffusivity"]["profile_kd"]["sets_mW"]["dJ<=3.84"][0][i]
               - JF["sites"][s]["with_diffusivity"]["bootstrap"]["kd_mW"][q])
           for s in S2 for i, q in ((0, "p2.5"), (1, "p97.5")))
block = max(abs(bb["probe_block"][s]["kd_mW"][q] - bb["production"][s]["kd_mW"][q]) for s in S2 for q in ("p2.5", "p97.5"))
chk = nb.Checks("Section 2.3:")
chk.add_text("albedos x K_d values", f"{len(m['A_grid'])} x {len(m['kd_grid_mW'])}", "25 x 94")
chk.add_text("albedo grid", f"{min(m['A_grid']):.3f}-{max(m['A_grid']):.3f}", "0.115-0.175")
chk.add_text("K_d grid (mW m^-1 K^-1)", f"{min(m['kd_grid_mW']):g}-{max(m['kd_grid_mW']):g}", "1-24")
chk.add_text("bootstrap repeats", str(m["n_boot"]), "1500")
chk.add("depth jitter (cm)", m["depth_jitter_cm"], "2.5", ".1f")
chk.add("profile likelihood vs bootstrap: 95% bounds agree to", prof, "0.02", ".2f")
chk.add("sensors grouped by probe: 95% bounds move by at most", block, "0.04", ".2f")
chk.show()

## Section 3.1: the models against each sensor (Table 1, Figure 3)
At its published values ($K_d=3.4$, $A=0.12$) the Hayne et al. (2017) model runs
warmer than the sensors. Refitting the albedo removes the offset; at Apollo 15
the global value then fits the sensors as well as the recalibrated $K_d$, but
at Apollo 17 it predicts a gradient two to six times the measured one. The
sensors also differ from one another: by probe, and by when their windows were
averaged.

In [ ]:
rows = []
for s in S2:
    h = LR["standard_albedo_0p12"][s]["hayne_as_published"]
    rows.append((s, "Hayne, published values", 0.12, 3.4, h["sensor_rmse_K"], h["sensor_bias_K"], np.mean(h["kappa_model_1e4_cm2_s"])))
    for key, name in (("global_3p4", "Hayne, K_d = 3.4, A refitted"), ("fit_with_diffusivity", "Hayne, joint fit"),
                      ("martinez_forward", "Martinez & Siegler")):
        r = FC[s][key]
        rows.append((s, name, r["A"], r.get("kd_mW"), r["rmse_K"], r["bias_K"], np.mean(r["kappa_model"]) if "kappa_model" in r else None))
display(pd.DataFrame(rows, columns=["site", "model (Table 1)", "A", "K_d", "RMSE (K)", "bias (K)", "kappa (1e-4 cm^2/s)"]).round(3))

In [ ]:
chk = nb.Checks("Table 1:")
P = {"A15": dict(pub=("1.77", "+1.53", "0.56", "6e-8"), glob=("0.138", "0.90", "+0.02", "0.56", "9e-7"),
                 fit=("0.136", "0.90", "0.00", "0.80"), ms=("0.91", "+0.14")),
     "A17": dict(pub=("2.09", "+1.97", "0.57", "5e-21"), glob=("0.143", "0.71", "+0.01", "0.56", "1e-14"),
                 fit=("0.137", "0.43", "0.00", "0.98"), ms=("0.62", "+0.05"))}
for s in S2:
    p, h = P[s], LR["standard_albedo_0p12"][s]["hayne_as_published"]
    chk.add(f"{s} published values: RMSE (K)", h["sensor_rmse_K"], p["pub"][0], ".2f")
    chk.add(f"{s} published values: bias (K)", h["sensor_bias_K"], p["pub"][1], "+.2f")
    chk.add(f"{s} published values: kappa", np.mean(h["kappa_model_1e4_cm2_s"]), p["pub"][2], ".2f")
    chk.add(f"{s} published values: p", LR["standard_albedo_0p12"][s]["global_3p4_at_standard_albedo"]["p"], p["pub"][3], ".0e")
    g = FC[s]["global_3p4"]
    chk.add(f"{s} K_d=3.4, A refitted: A", g["A"], p["glob"][0], ".3f")
    chk.add(f"{s} K_d=3.4, A refitted: RMSE (K)", g["rmse_K"], p["glob"][1], ".2f")
    chk.add(f"{s} K_d=3.4, A refitted: bias (K)", g["bias_K"], p["glob"][2], "+.2f")
    chk.add(f"{s} K_d=3.4, A refitted: kappa", np.mean(g["kappa_model"]), p["glob"][3], ".2f")
    chk.add(f"{s} K_d=3.4, A refitted: p", LR["sites"][s]["global_3p4_with_diffusivity"]["p"], p["glob"][4], ".0e")
    f = FC[s]["fit_with_diffusivity"]
    chk.add(f"{s} joint fit: A", f["A"], p["fit"][0], ".3f")
    chk.add(f"{s} joint fit: RMSE (K)", f["rmse_K"], p["fit"][1], ".2f")
    chk.add(f"{s} joint fit: bias (K)", f["bias_K"], p["fit"][2], ".2f")
    chk.add(f"{s} joint fit: kappa", np.mean(f["kappa_model"]), p["fit"][3], ".2f")
    chk.add(f"{s} Martinez & Siegler: RMSE (K)", FC[s]["martinez_forward"]["rmse_K"], p["ms"][0], ".2f")
    chk.add(f"{s} Martinez & Siegler: bias (K)", FC[s]["martinez_forward"]["bias_K"], p["ms"][1], "+.2f")
chk.show()

In [ ]:
ob = GE["sites"]["A17"]["observed"]
gt = FC["A17"]["global_residual_trend"]
al = [c for c in R["albedo_sensitivity"]["cases"] if c["site"] == "A17"
      and abs(c["bias_K"]) <= 1 and abs(c["surface_mean_K"] - c["surface_mean_obs_K"]) <= 5]
ratios = [GE["sites"]["A17"]["model_gradient_K_per_m"]["global_3p4"] / ob[k]["gradient_K_per_m"]
          for k in ("certified", "common_1974", "translated_1974", "thermoluminescence")]
chk = nb.Checks("Section 3.1, the A17 gradient (SI Table S2):")
chk.add("published values run warmer than the sensors, A15 (K)", LR["standard_albedo_0p12"]["A15"]["hayne_as_published"]["sensor_bias_K"], "1.5", ".1f")
chk.add("published values run warmer than the sensors, A17 (K)", LR["standard_albedo_0p12"]["A17"]["hayne_as_published"]["sensor_bias_K"], "2.0", ".1f")
chk.add("A17 global value: residual at 130 cm (K)", gt["shallow_K"], "-1.26", ".2f")
chk.add("A17 global value: residual at 234 cm (K)", gt["deep_K"], "+0.76", "+.2f")
chk.add("A17 gradient, global value (K/m)", GE["sites"]["A17"]["model_gradient_K_per_m"]["global_3p4"], "2.28", ".2f")
chk.add("A17 measured gradient, per-sensor windows", ob["certified"]["gradient_K_per_m"], "0.37", ".2f")
chk.add("   its standard error", ob["certified"]["se"], "0.23", ".2f")
chk.add_text("A17 measured gradient, common 1974 epoch", nb.span([ob["common_1974"]["gradient_K_per_m"], ob["translated_1974"]["gradient_K_per_m"]], ".2f"), "0.81-0.89")
chk.add("   its standard error", max(ob["common_1974"]["se"], ob["translated_1974"]["se"]), "0.2", ".1f")
chk.add("A17 drill-core gradient (Sehlke et al., 2026)", ob["thermoluminescence"]["gradient_K_per_m"], "1.0", ".1f")
chk.add("   its uncertainty", ob["thermoluminescence"]["se"], "0.5", ".1f")
chk.add_text("global gradient / measured: two to six times", f"{nb.rounded(min(ratios), '.0f')}-{nb.rounded(max(ratios), '.0f')}", "2-6")
chk.add_text("A17 global gradient under every albedo treatment that matches both measurements", nb.span([c["gradient_model_at_global"] for c in al], ".2f"), "2.26-2.29")
chk.add("Martinez & Siegler: A17 gradient (K/m)", GE["sites"]["A17"]["model_gradient_K_per_m"]["martinez"], "1.99", ".2f")
chk.add("A15 measured gradient (K/m)", GE["sites"]["A15"]["observed"]["certified"]["gradient_K_per_m"], "2.54", ".2f")
chk.add("   its standard error", GE["sites"]["A15"]["observed"]["certified"]["se"], "2.02", ".2f")
chk.add("A15 gradient, global value (K/m)", GE["sites"]["A15"]["model_gradient_K_per_m"]["global_3p4"], "3.06", ".2f")
chk.show()

Each sensor minus the joint fit, by probe, thermometer type (TG: gradient
bridge; TR: ring bridge) and the year its window ends. The offsets between the
probes and between the types are what the letter quotes.

In [ ]:
for s in S2:
    print(f"Apollo {s[1:]}")
    display(pd.DataFrame(FC[s]["sensor_residuals"]["sensors"]).round(2))
sr15, sr17 = FC["A15"]["sensor_residuals"], FC["A17"]["sensor_residuals"]
type_gap = [sr["type_mean_K"][f"probe{p}_TG"] - sr["type_mean_K"][f"probe{p}_TR"]
            for sr in (sr15, sr17) for p in (1, 2) if f"probe{p}_TR" in sr["type_mean_K"]]
chk = nb.Checks("Section 3.1, the sensors (Figure 3c, d):")
chk.add_text("A15 probe 2: below the fit by (K)", nb.span([-x for x in sr15["probe_range_K"]["probe2"]], ".1f"), "1.3-1.4")
chk.add("A15 probe 1: above the fit, least (K)", sr15["probe_range_K"]["probe1"][0], "0.15", ".2f")
chk.add("A15 probe 1: above the fit, most (K)", sr15["probe_range_K"]["probe1"][1], "1.0", ".1f")
chk.add_text("A17 probe 2 minus probe 1, both windows in 1974 (K)", nb.span([q["diff_K"] for q in sr17["probe2_minus_probe1_K"]["1974"]], ".2f"), "0.35-0.56")
chk.add_text("A17 probe 2 minus probe 1, both windows in 1977 (K)", nb.span([q["diff_K"] for q in sr17["probe2_minus_probe1_K"]["1977"]], ".2f"), "0.10-0.13")
chk.add("TG and TR of one A17 probe, same year, 10 cm apart: differ by at most (K)",
        max(abs(q["diff_K"]) for q in sr17["tg_minus_tr_same_year_K"]), "0.1", ".1f")
chk.add("TR read colder than TG of the same probe, on average by up to (K)", max(type_gap), "0.4", ".1f")
chk.show()

**Figure 3.** The models against each sensor.

In [ ]:
nb.show("fig_joint_sensors")

## Section 3.2: one measurement at a time (Figures 4, 5; SI Tables S7–S10)
On their own, the temperatures leave $K_d$ open or tie it to an assumed albedo;
the annual wave fixes it through how the wave fades and lags with depth.

**Sensor temperatures at a fixed albedo** (SI Table S7).

In [ ]:
cases = R["albedo_sensitivity"]["cases"]
const = {s: {round(c["A"], 4): c["kd_star_mW"] for c in cases if c["site"] == s and c["family"] == "constant"} for s in S2}
mapc = {s: next(c["kd_star_mW"] for c in cases if c["site"] == s and c["family"] == "angular" and c["f"] == 0.0) for s in S2}
laws = [c["kd_star_mW"] for c in cases if c["family"] == "angular" and c["f"] == 1.0] + \
       [e["kd_star_mW"] for e in R["albedo_anchor"]["published_laws_at_sites"]]
display(pd.DataFrame({s: pd.Series(const[s]) for s in S2}).rename_axis("fixed albedo").round(2).T)
chk = nb.Checks("Section 3.2, a fixed albedo (SI Table S7):")
for s, k012, kmap, lo, hi in (("A15", "4.6", "4.6", "2.9", "16"), ("A17", "5.8", "5.7", "5.6", "18")):
    scan = [v for a, v in const[s].items() if 0.08 <= a <= 0.16]
    chk.add(f"{s}: albedo 0.12", const[s][0.12], k012, ".1f")
    chk.add(f"{s}: map albedo of the site", mapc[s], kmap, ".1f")
    chk.add(f"{s}: albedo 0.08-0.16, lowest", min(scan), lo, ".1f")
    chk.add(f"{s}: albedo 0.08-0.16, highest", max(scan), hi, ".0f")
chk.add("published angular albedo laws, lowest", min(laws), "0.9", ".1f")
chk.add("published angular albedo laws, highest (grid edge)", max(laws), "27", ".0f")
chk.show()

**Sensor temperatures and the surface mean, with the albedo free** (Figure 4,
SI Tables S8, S9). The temperatures allow a long valley of solutions; this is
also why the global value passed the original ±5 K test.

In [ ]:
to = {s: JF["sites"][s]["temperature_only"] for s in S2}
tw = R["transient_warming"]["sites"]["A17"]["cases"]["step_1.6K"]["surface_as_measured"]["temperature_only"]
kt = R["joint_probe_checks"]["sites"]
chk = nb.Checks("Section 3.2, temperatures alone (SI Tables S8, S9):")
chk.add("A15: best fit", to["A15"]["best"]["kd_star_mW"], "3.7", ".1f")
chk.add_text("A15: 68% interval (profile likelihood)", nb.span(to["A15"]["profile_kd"]["sets_mW"]["dJ<=1.0"][0], ".1f"), "2.4-6.7")
chk.add("A17: bounded from below at (bootstrap 2.5%)", to["A17"]["bootstrap"]["kd_mW"]["p2.5"], "8.5", ".1f")
chk.add("A17, the warming after deployment removed: K_d", tw["kd_mW"], "7.4", ".1f")
chk.add("   95% lower", tw["interval95_mW"][0], "6.5", ".1f")
chk.add("   95% upper", tw["interval95_mW"][1], "9.3", ".1f")
for s, lo, hi in (("A15", "1.3", "6.5"), ("A17", "1.3", "10.3")):
    k = kt[s]["keihm_test_A_0.12"]["kd_pass_mW"]
    chk.add(f"{s}: the +-5 K test at A = 0.12 passes K_d from", k[0], lo, ".1f")
    chk.add(f"{s}:    to", k[1], hi, ".1f")
chk.show()

In [ ]:
nb.show("fig_joint_constraints")

**The annual wave alone** (Figure 5, SI Table S10). Measured again at every
sensor in the full record; modeled with the real sunlight of 1966–1977, with a
free size and timing factor per probe.

In [ ]:
aw = R["annual_wave"]["sites"]
st = R["annual_wave_surface_test"]["sites"]
display(pd.DataFrame({s: {v: round(aw[s]["variants"][v]["kd_mW"], 2) for v in aw[s]["variants"]} for s in S2})
        .rename_axis("annual-wave variant: K_d"))
unif = [aw[s]["uniform_kappa"][p]["amplitude"]["kappa"] for s in S2 for p in ("p1", "p2")]
scale = [st[s]["nominal"]["factor_at_best"][p]["scale"] for s in S2 for p in ("p1", "p2")]
lag = [st[s]["nominal"]["factor_at_best"][p]["lag_days"] for s in S2 for p in ("p1", "p2")]
chk = nb.Checks("Section 3.2, the annual wave (SI Table S10):")
chk.add_text("diffusivities from our amplitudes, uniform medium (1e-4 cm^2/s)",
             ", ".join(nb.rounded(x, ".2f") for x in unif), "0.85, 0.77, 1.01, 0.94")
chk.add_text("timing at the shallowest sensors right to within (days)", f"{nb.rounded(min(lag), '.0f')}-{nb.rounded(max(lag), '.0f')}", "1-12")
chk.add_text("the model wave is too large by a factor", nb.span([1 / x for x in scale], ".1f"), "1.5-2.5")
for s, (kd, lo, hi, dc) in (("A15", ("5.7", "5.3", "6.1", "190")), ("A17", ("5.6", "5.2", "6.1", "130"))):
    n = aw[s]["variants"]["nominal"]
    chk.add(f"{s}: K_d", n["kd_mW"], kd, ".1f")
    chk.add(f"{s}: 95% lower", n["ci95"][0], lo, ".1f")
    chk.add(f"{s}: 95% upper", n["ci95"][1], hi, ".1f")
    chk.add_text(f"{s}: Delta chi^2 of the global value", nb.rounded(n["dchi2_global"] / 10, ".0f") + "0", dc)
for v, val in (("amplitude_only", "5.6"), ("phase_only", "5.9"), ("langseth_pairs_amp", "4.8")):
    chk.add(f"A15: {v.replace('_', ' ')}", aw["A15"]["variants"][v]["kd_mW"], val, ".1f")
chk.show()

In [ ]:
nb.show("fig_joint_annual_wave")

## Section 3.3: all measurements together (the joint fit)
The values the letter adopts: 4.9 [4.3, 5.5] at Apollo 15 and 5.9 [5.3, 6.5]
mW m⁻¹ K⁻¹ at Apollo 17 (95% bootstrap intervals).

In [ ]:
rows = []
for s in S2:
    for fit in ("with_diffusivity", "temperature_only"):
        b, bt = JF["sites"][s][fit]["best"], JF["sites"][s][fit]["bootstrap"]
        rows.append((s, "sensors + surface mean + diffusivity" if fit == "with_diffusivity" else "sensors + surface mean only",
                     round(b["A"], 4), f'{bt["A"]["p2.5"]:.3f} - {bt["A"]["p97.5"]:.3f}',
                     round(b["kd_star_mW"], 2), f'{bt["kd_mW"]["p2.5"]:.2f} - {bt["kd_mW"]["p97.5"]:.2f}'))
display(pd.DataFrame(rows, columns=["site", "fitted to", "albedo A", "A, 95%", "K_d", "K_d, 95%"]))

In [ ]:
kobs = JF["meta"]["kappa_obs_langseth1976_table1"]
zsig = max(abs(km - o[2]) / o[3] for s in S2 for km, o in zip(FC[s]["fit_with_diffusivity"]["kappa_model"], kobs[s]))
va = R["joint_valley"]["A17"]["rows"]; r6, r7 = (next(r for r in va if r["kd_mW"] == k) for k in (6.0, 7.0))
sa = LR["standard_albedo_0p12"]
chk = nb.Checks("Section 3.3, the joint fit:")
for s, A, lo_a, hi_a, kd, lo, hi, ts in (("A15", "0.136", "0.128", "0.145", "4.9", "4.3", "5.5", "213.5"),
                                         ("A17", "0.137", "0.135", "0.140", "5.9", "5.3", "6.5", "216.6")):
    w = JF["sites"][s]["with_diffusivity"]
    chk.add(f"{s}: A*", w["best"]["A"], A, ".3f")
    chk.add(f"{s}: A* 95% lower", w["bootstrap"]["A"]["p2.5"], lo_a, ".3f")
    chk.add(f"{s}: A* 95% upper", w["bootstrap"]["A"]["p97.5"], hi_a, ".3f")
    chk.add(f"{s}: K_d*", w["best"]["kd_star_mW"], kd, ".1f")
    chk.add(f"{s}: K_d* 95% lower", w["bootstrap"]["kd_mW"]["p2.5"], lo, ".1f")
    chk.add(f"{s}: K_d* 95% upper", w["bootstrap"]["kd_mW"]["p97.5"], hi, ".1f")
    chk.add(f"{s}: modeled surface mean (K)", FC[s]["fit_with_diffusivity"]["surface_mean_K"], ts, ".1f")
chk.add_text("K_d* / global value", nb.span([best[s]["kd_star_mW"] / 3.4 for s in S2], ".1f"), "1.4-1.7")
chk.add("model diffusivities agree with the measured ones within (standard errors)", zsig, "1.3", ".1f")
chk.add("A17 valley: K_d 6 -> 7 improves the sensor misfit by (K)", r6["sensor_rmse_K"] - r7["sensor_rmse_K"], "0.04", ".2f")
chk.add_text("   and moves the diffusivity above the probes by (standard errors)", " and ".join(nb.rounded(x, ".1f") for x in r7["kappa_sigmas"]), "2.1 and 3.6")
chk.add_text("fitted albedo above 0.12 by", nb.span([best[s]["A"] - 0.12 for s in S2], ".3f"), "0.016-0.017")
for s, kd, bias, p in (("A15", "4.8", "1.35", "0.004"), ("A17", "5.6", "1.47", "2e-10")):
    chk.add(f"{s}, albedo held at 0.12: K_d*", sa[s]["kd_star_mW"], kd, ".1f")
    chk.add(f"{s}, albedo held at 0.12: sensors too warm by (K)", sa[s]["sensor_bias_K"], bias, ".2f")
    chk.add(f"{s}, albedo held at 0.12: p", sa[s]["fixed_vs_fitted_albedo"]["p"], p, ".3f" if s == "A15" else ".0e")
chk.show()

In [ ]:
st_ = LR["sites"]
pv = LR["diffusivity_uncertainty_doubled"]
chk = nb.Checks("Section 3.3, the global value is rejected:")
chk.add("A17: Delta J", st_["A17"]["global_3p4_with_diffusivity"]["delta_J"], "59.8", ".1f")
chk.add("A17: p", st_["A17"]["global_3p4_with_diffusivity"]["p"], "1e-14", ".0e")
chk.add("A15, temperatures alone: p", st_["A15"]["global_3p4_temperature_only"]["p"], "0.85", ".2f")
chk.add("A15: diffusivity at K_d = 3.4", np.mean(FC["A15"]["global_3p4"]["kappa_model"]), "0.56", ".2f")
chk.add("A15: Delta J", st_["A15"]["global_3p4_with_diffusivity"]["delta_J"], "24.1", ".1f")
chk.add("A15: p", st_["A15"]["global_3p4_with_diffusivity"]["p"], "9e-7", ".0e")
chk.add("Feng et al. (2020) 3.8, A15: p", st_["A15"]["published_global_values"]["feng2020"]["p"], "4e-4", ".0e")
chk.add("Feng et al. (2020) 3.8, A17: p", st_["A17"]["published_global_values"]["feng2020"]["p"], "8e-11", ".0e")
chk.add("diffusivity errors doubled, both sites: p", pv["pooled"]["p"], "4e-8", ".0e")
chk.add("diffusivity errors doubled, A15: p", pv["A15"]["p"], "0.02", ".2f")
chk.show()

In [ ]:
ob = GE["sites"]["A17"]["observed"]
tws = R["transient_warming"]["sites"]["A17"]["cases"]["step_1.6K"]["surface_as_measured"]
chk = nb.Checks("Section 3.3, the A17 gradient at the joint fit:")
chk.add("model gradient (K/m)", GE["sites"]["A17"]["model_gradient_K_per_m"]["joint_fit"], "1.32", ".2f")
chk.add("above the common-epoch value by (standard errors)", ob["common_1974"]["tension_sigma"]["joint_fit"], "2.5", ".1f")
chk.add("above the per-sensor value by (standard errors)", ob["certified"]["tension_sigma"]["joint_fit"], "4.2", ".1f")
chk.add_text("basal flux that would match those gradients (mW m^-2)", nb.span([ob["certified"]["qb_matching_mW"], ob["common_1974"]["qb_matching_mW"]], ".1f"), "3.6-9.5")
chk.add("measured gradient with the warming removed (K/m)", tws["gradient_observed_corrected_K_per_m"], "1.06", ".2f")
chk.show()

## Section 3.4: checks of the joint fit (Table 2, Figures 6, 7; SI Tables S5, S6, S12, S13)
Every choice and input changed in turn.

In [ ]:
sens = R["joint_fit_sensitivities"]["sites"]
rean = {s: [v["with_diffusivity"]["kd_star_mW"] if "with_diffusivity" in v else v["kd_star_mW"]
            for k, v in sens[s]["reanalysis"].items() if not k.startswith("zb_")] for s in S2}
cut = [abs((sens[s]["reanalysis"][k]["with_diffusivity"] if "with_diffusivity" in sens[s]["reanalysis"][k]
            else sens[s]["reanalysis"][k])["kd_star_mW"] - best[s]["kd_star_mW"]) for s in S2 for k in ("zb_70", "zb_90")]
record = [abs(v - best[s]["kd_star_mW"]) for s in S2 for v in rean[s] + [FC[s][f"drift_forward_{t}yr"]["kd_mW"] for t in (1, 2)]]
warm = max(abs(c["surface_as_measured"]["d_kd_mW"]) for s in S2 for k, c in R["transient_warming"]["sites"][s]["cases"].items()
           if k == "step_1.6K" or s == "A15")
chk = nb.Checks("Section 3.4, the sensors and the record:")
chk.add("A15: probe 1 only", FC["A15"]["probe1_only"]["kd_mW"], "4.99", ".2f")
chk.add("A17: probe 1 only", FC["A17"]["probe1_only"]["kd_mW"], "5.78", ".2f")
chk.add("A17: probe 2 only", FC["A17"]["probe2_only"]["kd_mW"], "5.83", ".2f")
for s, tg, tr in (("A15", "4.87", "4.86"), ("A17", "5.78", "5.80")):
    chk.add(f"{s}: TG only", FC[s]["split_TG"]["kd_mW"], tg, ".2f")
    chk.add(f"{s}: TR only", FC[s]["split_TR"]["kd_mW"], tr, ".2f")
chk.add("A15: deepest sensor held out, predicted to (K)", abs(FC["A15"]["holdout_deepest"]["residual_K"]), "0.18", ".2f")
chk.add("A17: deepest sensor held out, predicted to (K)", abs(FC["A17"]["holdout_deepest"]["residual_K"]), "0.26", ".2f")
chk.add("cut at 70 or 90 cm: K_d moves by at most", max(cut), "0.01", ".2f")
chk.add("windows, common 1974 window, 1-2 yr ahead: at most", max(record), "0.04", ".2f")
chk.add("warming after deployment removed: at most", warm, "0.06", ".2f")
chk.add("A17, albedo fixed, temperatures only: +1 K moves K_d by", FC["A17"]["fixed_albedo_uniform_bias"]["shift_plus1K_mW"], "+5.6", "+.1f")
chk.add("A17, albedo fixed, temperatures only: -1 K moves K_d by", FC["A17"]["fixed_albedo_uniform_bias"]["shift_minus1K_mW"], "-1.4", "+.1f")
chk.show()

In [ ]:
eb = R["joint_fit_sensitivities"]["error_budget"]
keys = ["sigma_stat", "sigma_solver", "sigma_Qb", "sigma_rho", "sigma_cp", "sigma_Ks",
        "sigma_zb", "sigma_thr", "sigma_window", "sigma_epoch", "total_quadrature"]
display(pd.DataFrame({s: [round(eb[s][k], 4) for k in keys] for s in S2}, index=keys).rename_axis("Table 2 (mW m^-1 K^-1)"))
cd = R["joint_chi_density"]["sites"]
chk = nb.Checks("Table 2, error budget:")
P = {"A15": ("0.30", "<0.05", "<0.05", "0.40", "0.15", "<0.05", "0.52", "0.45"),
     "A17": ("0.33", "<0.05", "0.15", "0.51", "0.15", "<0.05", "0.64", "0.55")}
for s in S2:
    e, p = eb[s], P[s]
    for (lab, key), val in zip((("statistical", "sigma_stat"), ("solver", "sigma_solver"), ("Q_b", "sigma_Qb"),
                                ("density", "sigma_rho"), ("c_p", "sigma_cp"), ("K_s", "sigma_Ks")), p[:6]):
        chk.add(f"{s}: {lab}", e[key], val, ".2f")
    for key in ("sigma_zb", "sigma_thr", "sigma_window", "sigma_epoch"):
        chk.add(f"{s}: {key.split('_')[1]}", e[key], "<0.05")
    chk.add(f"{s}: total (quadrature)", e["total_quadrature"], p[6], ".2f")
    chk.add(f"{s}: chi 2.2-3.2 (half range)", abs(cd[s]["chi_2.2"]["kd_star_mW"] - cd[s]["chi_3.2"]["kd_star_mW"]) / 2, p[7], ".2f")
    chk.add(f"{s}: H 3-10 cm", e["conditional"]["H"], "<0.05")
    chk.add(f"{s}: albedo form", max(abs(v - e["kd_star_mW"]) for v in e["conditional"]["albedo_form_kd_mW"].values()), "<0.05")
qb = {s: max(abs(v["with_diffusivity"]["kd_star_mW"] - best[s]["kd_star_mW"]) for k, v in sens[s]["variants"].items() if k.startswith("Qb_")) for s in S2}
chk.add("A15: basal flux 14-25 mW m^-2 moves K_d by at most (SI Table S12)", qb["A15"], "0.01", ".2f")
chk.add("A17: basal flux 10-18 mW m^-2 moves K_d by at most", qb["A17"], "0.25", ".2f")
chk.show()

In [ ]:
mc = R["joint_mcmc"]
od = R["own_diffusivity_fit"]["variants"]
own = [v for v in od if v != "langseth1976"]
chk = nb.Checks("Section 3.4, the posterior and our own diffusivities (SI Table S6):")
for s, (m50, lo, hi) in (("A15", ("4.85", "4.10", "5.67")), ("A17", ("5.94", "5.07", "6.86"))):
    k = mc[s]["posterior"]["kd_mW"]
    chk.add(f"{s}: posterior median", k["p50"], m50, ".2f")
    chk.add(f"{s}: posterior 95% lower", k["p2.5"], lo, ".2f")
    chk.add(f"{s}: posterior 95% upper", k["p97.5"], hi, ".2f")
for s, rng in (("A15", "4.9-6.1"), ("A17", "5.8-6.1")):
    chk.add_text(f"{s}: K_d with our own diffusivities", nb.span([od[v][s]["kd_star_mW"] for v in own], ".1f"), rng)
chk.add("our own diffusivities: largest p of the global value", max(od[v][s]["global_3p4"]["p"] for v in own for s in S2), "<1e-6")
chk.show()

**Every variant that uses the annual wave gives 4.5–6.6** (Figure 7). The rows
of Figure 7 are read from the results files by `make_joint_figures.variant_rows`.

In [ ]:
import sys
for p in ("pipeline/figures", "pipeline/compute", "."):
    if str(nb.CODE / p) not in sys.path:
        sys.path.insert(0, str(nb.CODE / p))
import make_joint_figures as mj
rows = mj.variant_rows(JF, FC, R["joint_fit_sensitivities"])
annual = [r for r in rows if r[0] != "One measurement at a time" or r[1].startswith("Annual wave")]
vals = [x for r in annual for s in S2 for x in ([r[2][s]["best"]] if r[2][s]["best"] is not None else []) + r[2][s]["pts"]]
display(pd.DataFrame([(g, lab, *(f'{d[s]["best"]:.2f}' if d[s]["best"] is not None else nb.span(d[s]["pts"], ".2f") for s in S2))
                      for g, lab, d in rows], columns=["group", "variant", "A15", "A17"]))
chk = nb.Checks("Section 3.4, Figure 7:")
chk.add_text("every variant that uses the annual wave", nb.span(vals, ".1f"), "4.5-6.6")
chk.show()

**Figure 6** (the posterior), **Figure 7** ($K_d$ from every variant),
**SI Figure S5** ($K_d$ under each changed input) and **SI Figure S6** (Diviner;
needs the Diviner tiles).

In [ ]:
for f in ("fig_joint_mcmc", "fig_joint_variants", "fig_joint_robustness", "fig_joint_diviner_closure"):
    nb.show(f)

## Section 4: discussion
What the values mean (4.1), whether the two sites differ (4.2) and what limits
the values (4.3, 4.4).

In [ ]:
jc = R["joint_mcmc"]["contrast_mW"]
ct = JF["contrast"]["with_diffusivity"]
po = LR["pooled"]
pc = R["joint_probe_checks"]["sites"]
awc = {v: aw["A17"]["variants"][v]["kd_mW"] - aw["A15"]["variants"][v]["kd_mW"] for v in aw["A15"]["variants"]}
chk = nb.Checks("Sections 4.1, 4.2:")
chk.add("A15: effective conductivity at the sensors", FC["A15"]["effective_conductivity_mW"]["mean"], "9.8", ".1f")
chk.add("A17: effective conductivity at the sensors", FC["A17"]["effective_conductivity_mW"]["mean"], "12.1", ".1f")
chk.add("A17 - A15, bootstrap median", ct["median"], "+1.0", "+.1f")
chk.add("   95% lower", ct["p2_5"], "+0.1", "+.1f")
chk.add("   95% upper", ct["p97_5"], "+1.9", "+.1f")
chk.add("pooled: one shared K_d", po["kd_shared_mW"], "5.3", ".1f")
chk.add("pooled: shared, Delta J", po["shared_vs_per_site"]["delta_J"], "5.3", ".1f")
chk.add("pooled: shared, p", po["shared_vs_per_site"]["p"], "0.02", ".2f")
chk.add("pooled: global 3.4, Delta J", po["global_vs_per_site"]["delta_J"], "83.7", ".1f")
chk.add("pooled: global 3.4, p", po["global_vs_per_site"]["p"], "7e-19", ".0e")
chk.add("A17 - A15, posterior median", jc["p50"], "+1.1", "+.1f")
chk.add("   95% lower", jc["p2.5"], "-0.1", "+.1f")
chk.add("   95% upper", jc["p97.5"], "+2.3", "+.1f")
for s, val in (("A15", "5.3, 4.5"), ("A17", "6.0, 5.3")):
    chk.add_text(f"{s}: K_d from each probe's diffusivity", ", ".join(nb.rounded(x["kd_mW"], ".1f") for x in pc[s]["per_probe_annual"]), val)
for s, val in (("A15", "3.7-4.4"), ("A17", "3.8-4.5")):
    chk.add_text(f"{s}: K_d from the transient diffusivities", nb.span(pc[s]["transient_kd_range_mW"], ".1f"), val)
chk.add("annual wave: the two sites agree to within", abs(awc["nominal"]), "0.1", ".1f")
chk.add_text("annual wave, amplitudes only: A17 - A15", f'{nb.rounded(awc["amplitude_only"], "+.1f")} to {nb.rounded(awc["langseth_pairs_amp"], "+.1f")}', "+0.6 to +0.8")
chk.add("annual wave, delays only: A17 - A15", awc["phase_only"], "+0.1", "+.1f")
oc = [R["own_diffusivity_fit"]["variants"][v]["contrast_mW"]["best"] for v in own]
chk.add_text("our own diffusivities in the joint fit: A17 - A15", f"{nb.rounded(min(oc), '+.1f')} to {nb.rounded(max(oc), '+.1f')}", "-0.4 to +1.2")
chk.show()

In [ ]:
chis = ("chi_2.2", "chi_2.45", "chi_2.95", "chi_3.2")
gl = LR["sites"]
mq = R["joint_mcmc"]["A17"]["posterior"]["Qb_mW"]
amp = {p: aw["A15"]["uniform_kappa"][p] for p in ("p1", "p2")}
surf = R["annual_wave_surface_test"]["sites"]
twc = R["transient_warming"]["sites"]
chk = nb.Checks("Sections 4.3, 4.4:")
for s, (k22, k32) in (("A15", ("5.4", "4.5")), ("A17", ("6.4", "5.3"))):
    chk.add(f"{s}: chi = 2.2", cd[s]["chi_2.2"]["kd_star_mW"], k22, ".1f")
    chk.add(f"{s}: chi = 3.2", cd[s]["chi_3.2"]["kd_star_mW"], k32, ".1f")
chk.add("global value at every chi 2.2-3.2: largest p", max(gl[s]["global_3p4_chi_density"][c]["p"] for s in S2 for c in chis), "<0.0002")
chk.add_text("fitted albedo over chi 2.2-3.2", nb.span([cd[s][c]["A"] for s in S2 for c in chis], ".2f"), "0.09-0.17")
chk.add_text("chi = 1.5: albedo at the edge (no physical albedo fits)", str(all(cd[s]["chi_1.5"]["A"] <= 0.0 for s in S2)), "True")
chk.add("A15: measured site density, K_d*", cd["A15"]["rho_site"]["kd_star_mW"], "4.9", ".1f")
chk.add("A17: measured site density, K_d*", cd["A17"]["rho_site"]["kd_star_mW"], "6.3", ".1f")
chk.add("A17 posterior basal flow, median (mW m^-2)", mq["p50"], "12", ".0f")
chk.add("   95% lower", mq["p2.5"], "8", ".0f")
chk.add("   95% upper", mq["p97.5"], "17", ".0f")
chk.add_text("A15: the delay implies a diffusivity higher than the fading does by (%)",
             nb.span([100 * (amp[p]["phase"]["kappa"] / amp[p]["amplitude"]["kappa"] - 1) for p in amp], ".0f"), "21-25")
chk.add_text("near-surface layer changed: annual-wave factors", nb.span([v["factor_at_best"][p]["scale"] for s in S2 for v in surf[s].values() for p in ("p1", "p2")], ".2f"), "0.34-0.78")
chk.add("   the annual-wave K_d moves by at most", max(abs(v["kd_mW"] - surf[s]["nominal"]["kd_mW"]) for s in S2 for v in surf[s].values()), "0.12", ".2f")
chk.add("A17, a darkening twice as large (3.5 K step): K_d lower by", -twc["A17"]["cases"]["step_3.5K"]["surface_as_measured"]["d_kd_mW"], "0.45", ".2f")
chk.show()

---
# 5. Summary

In [ ]:
ok = nb.summary()
print()
print("Every value matches the paper." if ok else "Some values differ from the paper: see the list above.")